# Задание 1

(**NB.** для запуска примеров кода нужен Python версии не ниже **3.10**).

Есть следующий код для [автоматического дифференцирования](https://en.wikipedia.org/wiki/Automatic_differentiation), в котором используются особенности системы типов языка `Python`: 

In [2]:
from dataclasses import dataclass
from typing import Union, Callable
from numbers import Number

@dataclass
class Dual:
    value: float
    d: float

    def __add__(self, other: Union["Dual", Number]) -> "Dual":
         match other:
            case Dual(o_value, o_d):
                return Dual(self.value + o_value, self.d + o_d)
            case Number():
                return Dual(float(other) + self.value, self.d)

    def __mul__(self, other: Union["Dual", Number]) -> "Dual":
         match other:
            case Dual(o_value, o_d):
                return Dual(self.value * o_value, self.value * o_d + self.d * o_value)
            case Number():
                return Dual(float(other) * self.value, float(other) * self.d)    

    __rmul__ = __mul__  # https://docs.python.org/3/reference/datamodel.html#object.__mul__
    __radd__ = __add__  # https://docs.python.org/3/reference/datamodel.html#object.__radd__
 

def diff(func: Callable[[float], float]) -> Callable[[float], float]:
    return lambda x: func(Dual(x, 1.0)).d 

Поддерживаются две операции - сложение и умножение. Применить можно так:

In [3]:
# Функция, которую будем дифференцировать
def f(x: float) -> float:
    return 5 * x * x + 2 * x + 2

f_diff = diff(f)

# значение производной в точке x = 2
f_diff(2)

22.0

## Задание 1.1 (5 баллов)

Какие недостатки вы видите в данной реализации? Реализуйте поддержку (полностью самостоятельно или модифицируя приведенный код):
- [унарных операций](https://docs.python.org/3/reference/datamodel.html#object.__neg__) 
- деления
- возведения в степень

Каким образом можно проверить корректность решения?  Реализуйте достаточный, по вашему мнению, набор тестов.

In [29]:
from dataclasses import dataclass
from typing import Callable
from numbers import Number
from math import log 

@dataclass(slots=True)
class Dual:
    value: float
    d: float

    def __add__(self, other: "Dual" | Number) -> "Dual":
        match other:
            case Dual(o_value, o_d):
                return Dual(self.value + o_value, self.d + o_d)
            case Number():
                return Dual(float(other) + self.value, self.d)

    def __mul__(self, other: "Dual" | Number) -> "Dual":
        match other:
            case Dual(o_value, o_d):
                return Dual(self.value * o_value, self.value * o_d + self.d * o_value)
            case Number():
                other_float = float(other)
                return Dual(other_float * self.value, other_float * self.d)

    def __sub__(self, other: "Dual" | Number) -> "Dual":
        match other:
            case Dual(o_value, o_d):
                return Dual(self.value - o_value, self.d - o_d)
            case Number():
                return Dual(self.value - float(other), self.d)

    def __rsub__(self, other: "Dual" | Number) -> "Dual":
        match other:
            case Dual(o_value, o_d):
                return Dual(o_value - self.value, o_d - self.d)
            case Number():
                return Dual(float(other) - self.value, -self.d)

    def __truediv__(self, other: "Dual" | Number):
        match other:
            case Dual(o_value, o_d):
                if o_value == 0:
                    raise ValueError("Деление на дуальное число с нулевым значением")
                return Dual(self.value / o_value, (self.d * o_value - self.value * o_d) / (o_value ** 2))
            case Number():
                other_float = float(other)
                if other_float == 0:
                    raise ValueError("Деление на ноль")
                return Dual(self.value / other_float, self.d / other_float)

    def __rtruediv__(self, other: "Dual" | Number) -> "Dual":
        match other:
            case Dual():
                return other.__truediv__(self)
            case Number():
                if self.value == 0:
                    raise ValueError("Деление на ноль")
                other_float = float(other)
                return Dual(other_float / self.value, -other_float * self.d / (self.value ** 2))

    def __pow__(self, n: int | float) -> "Dual":
        if n == 0:
            return Dual(1.0, 0.0)
        # Степень n <= 0 при нулевом основании дает математическую неопределенность
        # Степень 0 < n < 1 при нулевом основании дает деление на ноль
        if self.value == 0 and n < 1:
            raise ValueError("Производна не определена")
        return Dual(self.value ** n, self.d * n * self.value ** (n - 1))

    def __rpow__(self, base: int | float) -> "Dual":
        # Для вычисления производной a^x, a должно быть строго больше нуля
        if base <= 0:
            raise ValueError("Основание степени должно быть строго больше нуля")
        return Dual(base ** self.value, self.d * base ** self.value * log(base))

    def __pos__(self) -> "Dual":
        return Dual(+self.value, +self.d)

    def __neg__(self) -> "Dual":
        return Dual(-self.value, -self.d)

    def __abs__(self) -> "Dual":
        if self.value == 0:
            raise ValueError("Производная модуля в нуле не определена")
        sign = 1.0 if self.value > 0 else -1.0
        return Dual(abs(self.value), sign * self.d)

    def __invert__(self) -> "Dual":
        return Dual(self.d, self.value)
        
    def __repr__(self):
        return f"Dual({self.value}, {self.d})"

    __rmul__ = __mul__  # https://docs.python.org/3/reference/datamodel.html#object.__mul__
    __radd__ = __add__  # https://docs.python.org/3/reference/datamodel.html#object.__radd__


def diff(func: Callable[[float], float]) -> Callable[[float], float]:
    return lambda x: func(Dual(x, 1.0)).d 

***ТЕСТЫ***

In [5]:
import ipytest

ipytest.autoconfig()

In [7]:
%%ipytest
from math import isclose
import pytest


# __add__
@pytest.mark.parametrize(
    "a, b, expected_value, expected_d",
    [
        (Dual(1.0, 4.5), Dual(5.7, 0.5), 6.7, 5.0),
        (Dual(1.0, 4.5), 5.7, 6.7, 4.5),
        (23.3, Dual(9.9, 0.0), 33.2, 0.0),
    ]
)
def test_add_various_types(a, b, expected_value, expected_d):
    result = a + b
    assert isinstance(result, Dual)
    assert result.value == pytest.approx(expected_value)
    assert result.d == pytest.approx(expected_d)


# __mul__
@pytest.mark.parametrize(
    "a, b, expected_value, expected_d",
    [
        (Dual(1.0, 4.5), Dual(5.7, 0.5), 5.7, 26.15),
        (Dual(1.0, 4.5), 5.0, 5.0, 22.5),
        (5.0, Dual(5.0, 3.4), 25.0, 17.0),
    ]
)
def test_mul_various_types(a, b, expected_value, expected_d):
    result = a * b
    assert isinstance(result, Dual)
    assert result.value == pytest.approx(expected_value)
    assert result.d == pytest.approx(expected_d)


# __truediv__
@pytest.mark.parametrize(
    "a, b, expected_value, expected_d",
    [
        (Dual(10.0, 40.0), Dual(5.0, 8.0), 2.0, 4.8),
        (Dual(10.0, 40.0), 8.0, 1.25, 5.0),
        (20.0, Dual(5.0, 8.0), 4.0, -6.4),
    ]
)
def test_div_various_types(a, b, expected_value, expected_d):
    result = a / b
    assert isinstance(result, Dual)
    assert result.value == pytest.approx(expected_value)
    assert result.d == pytest.approx(expected_d)

def test_div_exceptions():
    with pytest.raises(ValueError):
        Dual(1.0, 1.0) / 0.0

    with pytest.raises(ValueError):
        30.0 / Dual(0.0, 3.0)


# __pow__
@pytest.mark.parametrize(
    "a, b, expected_value, expected_d",
    [
        (Dual(2.0, 5.0), 2, 4.0, 20.0),
        (3.0, Dual(3.0, 1.0), 27.0, 29.66253),
    ]
)
def test_pow_various_types(a, b, expected_value, expected_d):
    result = a ** b
    assert isinstance(result, Dual)
    assert result.value == pytest.approx(expected_value)
    assert result.d == pytest.approx(expected_d)

def test_pow_exceptions():
    with pytest.raises(ValueError):
        0.0 ** Dual(3.0, 2.0)


# __neg__
res = -Dual(1.0, 1.0)
assert res.value == -1.0, f"Ожидалось -1.0, получено {res.value}"
assert res.d == -1.0, f"Ожидалось -1.0, получено {res.d}"


# __abs__
res = abs(Dual(-5.0, 2.0))
assert res.value == 5.0, f"Ожидалось 5.0, получено {res.value}"
assert res.d == -2.0, f"Ожидалось -2.0, получено {res.d}"

with pytest.raises(ValueError):
    abs(Dual(0.0, 5.0))


# __invert__
res = ~Dual(-2.0, 5.0)
assert res.value == 5.0, f"Ожидалось 5.0, получено {res.value}"
assert res.d == -2.0, f"Ожидалось -2.0, получено {res.d}"

.............                                                                                [100%]
13 passed in 0.05s


## Задание 1.2 (7 баллов)
Придумайте способ и реализуйте поддержку функций:
- `exp()`
- `cos()`
- `sin()`
- `log()`

Добавьте соответствующие тесты

In [84]:
import math

def dual_exp(other: Dual | Number) -> Dual | float:
    match other:
        case Dual(o_value, o_d):
            return Dual(math.exp(o_value), o_d * math.exp(o_value))
        case Number():
            return math.exp(float(other))

def dual_sin(other: Dual | Number) -> Dual | float:
    match other:
        case Dual(o_value, o_d):
            return Dual(math.sin(other.value), other.d * math.cos(other.value))
        case Number():
            return math.sin(float(other))

def dual_cos(other: Dual | Number) -> Dual | float:
    match other:
        case Dual(o_value, o_d):
            return Dual(math.cos(other.value), -other.d * math.sin(other.value))
        case Number():
            return math.cos(float(other))

def dual_log(other: Dual | Number) -> Dual | float:
    value = other.value if isinstance(other, Dual) else float(other)

    if value <= 0:
        raise ValueError("Производная не определена при x <= 0")

    match other:
        case Dual(o_value, o_d):
            return Dual(math.log(value), o_d / value)
        case Number():
            return math.log(value)

***ТЕСТЫ***

In [85]:
from math import isclose, exp as math_exp, sin as math_sin, cos as math_cos, log as math_log

# EXP
res = dual_exp(Dual(3.0, 2.0))
assert isclose(res.value, math_exp(3.0))
assert isclose(res.d, 2.0 * math_exp(3.0))

assert isclose(dual_exp(3), math_exp(3))

# SIN
res = dual_sin(Dual(3.0, 2.0))
assert isclose(res.value, math_sin(3.0))
assert isclose(res.d, 2.0 * math_cos(3.0))

assert isclose(dual_sin(3), math_sin(3))

# COS
res = dual_cos(Dual(3.0, 2.0))
assert isclose(res.value, math_cos(3.0))
assert isclose(res.d, -2.0 * math_sin(3.0))

assert isclose(dual_cos(3), math_cos(3))

# LOG
res = dual_log(Dual(3.0, 2.0))
assert isclose(res.value, math_log(3.0))
assert isclose(res.d, 2.0 / 3.0)

assert isclose(dual_log(3), math_log(3))

## Задание 1.3 (2 балла)

Воспользуйтесь методами **численного** дифференцирования для "проверки" работы кода на нескольких примерах. Используйте библиотечный метод или реализуйте какой-нибудь метод численного дифференцирования самостоятельно (**+1 балл**)

In [94]:
# Метод центральной разности
def numerical_derivative(f: Callable[[float], float], x: float, h=1e-8) -> float:
    return (f(x + h) - f(x - h)) / (2 * h)

def f(x: float) -> float:
    return 5 * x * x + 2 * x + 2

f_diff = diff(f)

print("Численное дифф-е:", numerical_derivative(f, 2.))

print("Автоматическое дифф-е:", f_diff(2.))

Численное дифф-е: 22.000000043931323
Автоматическое дифф-е: 22.0


In [95]:
def f(x: float) -> float:
    return dual_sin(x) * dual_log(5 * x) + 2 * x ** 3 + dual_exp(x) + 1

f_diff = diff(f)

print("Численное дифф-е:", numerical_derivative(f, 7.6))

print("Автоматическое дифф-е:", f_diff(7.6))

Численное дифф-е: 2345.7972247342695
Автоматическое дифф-е: 2345.7972322815726


In [96]:
def f(x: float) -> float:
    return 3 * dual_cos(x / 2) ** 2 * dual_log(x / 3) / (2 * dual_exp (2 * x))

f_diff = diff(f)

print("Численное дифф-е:", numerical_derivative(f, 4.7))

print("Автоматическое дифф-е:", f_diff(4.7))

Численное дифф-е: -1.4129113833506301e-05
Автоматическое дифф-е: -1.4129114189357439e-05


## Задание 1.4 (6 баллов)

Необходимо разработать систему автоматического тестирования алгоритма дифференцирования в следующем виде:
- реализовать механизм генерации "случайных функций" (например, что-то вроде такого: $f(x) = x + 5 * x - \cos(20 * \log(12 - 20 * x * x )) - 20 * x$ )
- сгенерировать достаточно большое число функций и сравнить результаты символьного и численного дифференцирования в случайных точках 

Генерацию случайных функций можно осуществить, например, двумя путями. 
1. Генерировать функцию в текстовом виде, зачем использовать встроенную функцию [eval](https://docs.python.org/3/library/functions.html#eval)

```python
func = eval("lambda x: 2 * x + 5")
assert func(42) == 89 
```

2. Использовать стандартный модуль [ast](https://docs.python.org/3/library/ast.html), который позволяет во время выполнения программы манипулировать [Абстрактным Синтаксическим Деревом](https://ru.wikipedia.org/wiki/%D0%90%D0%B1%D1%81%D1%82%D1%80%D0%B0%D0%BA%D1%82%D0%BD%D0%BE%D0%B5_%D1%81%D0%B8%D0%BD%D1%82%D0%B0%D0%BA%D1%81%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5_%D0%B4%D0%B5%D1%80%D0%B5%D0%B2%D0%BE).
Например, выражение 

```python
func = lambda x: 2 * x + 5
```

Можно запрограммировать с помощью кода:

```python

expr = ast.Expression(
    body=ast.Lambda(
        args=ast.arguments(
            args=[
                ast.arg(arg='x')
            ],
            posonlyargs=[],
            kwonlyargs=[],
            kw_defaults=[],
            defaults=[]
        ),
        body=ast.BinOp(
            left=ast.BinOp(
                left=ast.Constant(value=2),
                op=ast.Mult(),
                right=ast.Name(id='x', ctx=ast.Load())
            ),
            op=ast.Add(),
            right=ast.Constant(value=5)
        )
    )
)

ast.fix_missing_locations(expr)

func = eval(compile(expr, filename="", mode="eval"))

assert func(42) == 89
```

При реализации нужно учитывать области допустимых значений функций.

In [108]:
import random

SAFE_MATH = {
    'sin': dual_sin,
    'cos': dual_cos,
    'exp': dual_exp,
    'log': dual_log,
    'abs': abs
}

def generate_expression(depth: int) -> str:
    if depth == 0:
            return "x"

    node_type = random.choice(['binary', 'unary'])

    if node_type == 'binary':
        op = random.choice(['+', '-', '*'])
        left = generate_expression(depth - 1)
        right = generate_expression(depth - 1)
        return f"({left} {op} {right})"
    
    else:
        op = random.choice(['sin', 'cos', 'exp', 'log', 'abs'])
        arg = generate_expression(depth - 1)
        return f"{op}({arg})"

def build_function(expression_string: str):
    return eval(f"lambda x: {expression_string}", SAFE_MATH)


In [109]:
from math import isclose

counter = 0
total_tests = 100
errors_log = []

for _ in range(total_tests):
    random_expr = generate_expression(3)
    f = build_function(random_expr)
    
    f_diff = diff(f) 
    test_x = random.uniform(0.1, 5.0)

    try:
        val_AD = f_diff(test_x)
        val_numerical = numerical_derivative(f, test_x)
        
        if isclose(val_AD, val_numerical, rel_tol=1e-4, abs_tol=1e-4):
            counter += 1
        else:
            errors_log.append(f"Несовпадение: {random_expr} при x={test_x:.2f}. AD: {val_AD:.4f}, Num: {val_numerical:.4f}")
            
    except Exception as e:
        errors_log.append(f"Ошибка {type(e).__name__} в выражении: {random_expr} | {e}")

print(f"Успешных совпадений: {counter} из {total_tests}")
if errors_log:
    print("\nОшибки/несовпадения:")
    for err in errors_log:
        print("-", err)


Успешных совпадений: 92 из 100

Ошибки/несовпадения:
- Ошибка ValueError в выражении: log(log(cos(x))) | Производная не определена при x <= 0
- Ошибка ValueError в выражении: (((x * x) * (x - x)) * log(cos(x))) | Производная не определена при x <= 0
- Ошибка ValueError в выражении: (((x + x) * exp(x)) + abs((x - x))) | Производная модуля в нуле не определена
- Ошибка ValueError в выражении: log(cos(exp(x))) | Производная не определена при x <= 0
- Ошибка ValueError в выражении: (abs((x - x)) + (sin(x) - log(x))) | Производная модуля в нуле не определена
- Ошибка ValueError в выражении: (exp(abs(x)) - abs((x - x))) | Производная модуля в нуле не определена
- Ошибка ValueError в выражении: (abs((x - x)) * ((x * x) + cos(x))) | Производная модуля в нуле не определена
- Ошибка ValueError в выражении: log((sin(x) * abs(x))) | Производная не определена при x <= 0


## Задание 1.5 (7 баллов)

Реализуйте поддержку функций нескольких аргументов. Например

```python
def f(x: float, y: float, z: float) -> float:
    return x * y + z - 5 * y  


f_diff = diff(f)

f_diff(10, 10, 10) # = [10, 5, 1]
```

In [ ]:
# ваш код